## In this tutorial we will use transformers to classify Ising model that you have done with logistic regression and SVM in your homework 1
We use the same dataset maintained by Pankajm and loaded the same way as before.

This code is from ChatGPT and modified by Gang Xu for the PSI scientific machine learning couse.

### Import libraries <span style="color:blue"> (No need to change)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pickle
from matplotlib import cm
import time
import torch as tc
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, random_split, DataLoader

### Read data <span style="color:blue"> (No need to change)

In [ ]:
#This is gratefully borrowed with permission from the notebooks maintained by P. Mehta.

######### LOAD DATA
######### The data consists of 16*10000 samples taken in T=np.arange(0.25,4.0001,0.25):
data_file_name = 'Ising2DFM_reSample_L40_T=All.pkl'
######### The labels are obtained from the following file:
label_file_name = 'Ising2DFM_reSample_L40_T=All_labels.pkl'


############ DATA
with open(data_file_name, 'rb') as pickle_file:
#r=read b=binary pickle must be read in binary mode and needs to be open
# with... as... will automatically close the file after opening it is safer
    data = pickle.load(pickle_file) # pickle reads the file and returns the Python object (1D array, compressed bits) and store in data

#### Decompress array and reshape for convenience
data = np.unpackbits(data).reshape(-1, 1600)
#data has byte (8bits) unpackbits unpack it into 8 bits return a bunch of 0 and 1s
#-1: figure out how many rows there are, each row has 1600=40*40 bits The length of the lattice is 40
data=data.astype('int')
#now convert the datatype to integer

#### map 0 state to -1 (Ising variable can take values +/-1)
data[np.where(data==0)]=-1
# np.where(data==0) find all indices where data is 0

###### READ LABELS (convention is 1 for ordered states and 0 for disordered states)
with open(label_file_name, 'rb') as pickle_file:
    labels = pickle.load(pickle_file) # pickle reads the file and returns the Python object (here just a 1D array with the binary labels)
print(data.shape) # the shape of the features
print(np.unique(labels)) # the unique labels

### Turn the data into something pytorch like<span style="color:blue"> (No need to change)

In [ ]:
X = tc.tensor(data, dtype=tc.float32)
y = tc.tensor(labels, dtype=tc.long)

# reshape to images: (N, 1, 40, 40)
X = X.view(-1, 1, 40, 40)

### Make dataloaders as in CNN<span style="color:blue"> (No need to change)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if tc.cuda.is_available() else "cpu"

dataset = TensorDataset(X, y)
train_size = int(0.8*len(dataset))
test_size = len(dataset) - train_size
train_ds, test_ds = tc.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=8)

### Making a patch embedding class transformer layer will use later <span style="color:blue"> (No need to change)

We divide the image into smaller pieces (10*10 e.g.) and flatten into a vector and project it into an embedding space. we call this new vector a token
The output is in the format of (N_samples (per batch)), num_patches, embed_dim) which still have all the information as before.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class PatchEmbedding(nn.Module):
    def __init__(self, patch_size=10, in_ch=1, embed_dim=64): #### dividing each image into 10*10=100 pixels each patches, for us the input channel is 1
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Linear(in_ch*patch_size*patch_size, embed_dim)
        #### each patch is then flattened to 100d vector and embedded to 64d vector (token), kind of like convolution kernel with stride size same as kernel size, but output is not featuremap but a bunch of tokens
        ### in our case, 16 tokens
    def forward(self, x):
        # x: (B, 1, 40, 40)
        B, C, H, W = x.shape
        patches = x.unfold(2, self.patch_size, self.patch_size)\
                   .unfold(3, self.patch_size, self.patch_size)
                   ### 2: split along height 3:split along width
        # patches: (B, 1, num_patches_H, num_patches_W, patch_H, patch_W) (B, 1, 4,4,10,10)
        patches = patches.contiguous().view(B, -1, C*self.patch_size*self.patch_size)
        # patches: (B, num_patches, patch_size*patch_size) -1:figure out how many patches are there and flatten patches to a vector
        embeddings = self.proj(patches)  # same embed_dim defined in the class
        return embeddings

### The transformer<span style="color:blue"> (No need to change)

Most of the interesting things happened in the transformerencoderlayer. We take this (B, num_patches, embed_dim) object coming out of the embedding
to make Q V K's by multiplying them with embed_dim square matrices. then divide into 4 different heads in this case each of them will recombine to the attention object (this part is really like transformer)
and we concate them together to form some shape just like the input. And the data has gone through transformation. 

In [ ]:
class IsingTransformer(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4, num_layers=2, num_classes=2): ## same embed_dim from the embedding class before
  ###  the model computes 4 separate attention mechanisms in parallel: 4 heads each head gets 16 dimensions small subspace and then combined
  ### so heads to be a divisor of embed_dimensions. evenly split between heads
  ### num_layers: This is the number of stacked Transformer encoder blocks.
  ### num_classes: output classes as before not tunable for our problem can change heads and layers to increase complexity
        super().__init__()
        self.patch_embed = PatchEmbedding(embed_dim=embed_dim) ### call the class just defined
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, batch_first=True
        )### This creates one Transformer encoder block. it splits into head and computes QKV projections (W_Q etc is dim*dim matrix weights)
        ### Q=x@W_Q with dim (B, num_patch, embed_dim) then separate into heads (B, num_patch, head, head_dim)
        ### then Q[:,:,0,:] is the head0 Q etc compute attention_head with softmax(QK^T/\sqr{heads})V for each head, this has dimension (B, num_patch, head_dim) as output of attention
        ### the final output will concate the heads attention together so it is still (B, num_patch, embed_dim)
        ### so basically the input is manipulated by the heads and the output of the same shape comes out and somehow more information is learned
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers) ### encoder_layer is now inside some self_, I am happy
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)  # 
        x = self.transformer(x)  # 
        x = x.mean(dim=1)         # 
        x = self.fc(x)            # 
        return x

### Train as before. This time we only train 5 epoch!

In [ ]:
model = IsingTransformer().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = tc.optim.Adam(model.parameters(), lr=1e-3)
tc.cuda.empty_cache() ### clear some precious memory used before, why not?
epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

### Evaluate the model as before. Also use transformers to detect the phase transition temperature before. 

In [ ]:
########### student write code


